# Engine calibration (Utente 1)

Goal: estimate the weights in `sentinel/model.py` (`PESI`) and the behavioural assumptions in `sentinel/profiles.py` from sources **relevant to scams against seniors**, then show the ROC curve and the sensitivity of the threshold.

**Sources to use**
- Swiss police crime statistics (FSO / UST): fraud offences, yearly trend
- NCSC reports on fake police officer and fake grandchild calls
- Local news (tio.ch, RSI): typical amounts and the script of the scams

**Do not use** sources on corporate data breaches (VERIS, IBM Cost of a Data Breach): they measure a different phenomenon.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np, pandas as pd
from sentinel import profiles, model, economics

## 1. Parameters from the sources
For each assumption write the value **and** the source next to it. Today these are placeholders:

| Assumption | Where in the code | Current value |
|---|---|---|
| Customers defrauded per year | `PREVALENZA_REALE` in the simulation | 1% |
| Typical external scam amount | `_operazioni_frode` | log-normal, median CHF 10'000 |
| Large legitimate cash withdrawals | `PRESET['anziano'].prelievi_straordinari` | 2 per year |
| Customers with a power-of-attorney holder | `prob_procuratore` | 15% |

In [ ]:
parametri = {
    # 'prevalenza_reale': (0.01, 'source'),
}

## 2. Calibration on a simulated book

In [ ]:
port = profiles.genera_portafoglio(n_clienti=300, prob_frode=0.3, seed=0)
scored = economics.punteggia_portafoglio(port)
curve, threshold = economics.calibra_soglia(scored)
threshold

## 3. ROC and sensitivity of the threshold
How does the threshold move with the cost of a false alarm (disturbing the contact) and with the fraud prevalence?

In [ ]:
in_scope = scored[scored['canale'] != 'carta']
roc, auc = economics.curva_roc(in_scope.loc[~in_scope.frode_reale, 'punteggio'], in_scope.loc[in_scope.frode_reale, 'punteggio'])
print('AUC', round(auc, 3))
for c in [1, 5, 20]:
    for prev in [0.005, 0.01, 0.02]:
        print(c, prev, economics.calibra_soglia(scored, prevalenza_reale=prev, costo_falso_allarme=c)[1])

## 4. (Advanced) Cross-check with an Isolation Forest